# Part III · Lesson 1 — Give the model something to read

## Import the document and model helpers

In [ ]:
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model
from langchain_core.documents import Document
from langchain_core.messages import SystemMessage, HumanMessage
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_openai import OpenAIEmbeddings

## Connect the two kinds of model

In [ ]:
load_dotenv()
# Groq generates answers; OpenAI creates search embeddings.
model = init_chat_model(
    "openai/gpt-oss-120b", model_provider="groq", reasoning_effort="low",
)
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

## Create a tiny knowledge collection

In [ ]:
sample_texts = [
    "LangGraph connects Python functions into a stateful workflow using nodes and edges.",
    "RAG retrieves relevant documents and adds their text to a prompt before generating an answer.",
    "A vector store keeps embeddings with their source text so similar material can be searched.",
    "The application executes tools requested by a model and returns the results to the model.",
]
documents = [
    Document(page_content=text, metadata={"source": f"course-note-{index}"})
    for index, text in enumerate(sample_texts, start=1)
]

## Make the notes searchable

In [ ]:
vector_store = InMemoryVectorStore.from_documents(documents, embeddings)
retriever = vector_store.as_retriever(search_kwargs={"k": 2})

## Describe the four-field state

In [ ]:
from typing_extensions import TypedDict
from langgraph.graph import StateGraph, START, END

class RagState(TypedDict):
    question: str
    documents: list[Document]
    answer: str
    needs_retrieval: bool

## Decide whether to search the notes

In [ ]:
def decide_retrieval(state: RagState) -> dict:
    keywords = ["what", "how", "explain", "describe", "tell me"]
    needs_retrieval = any(word in state["question"].lower() for word in keywords)
    return {"needs_retrieval": needs_retrieval}

def should_retrieve(state: RagState) -> str:
    return "retrieve" if state["needs_retrieval"] else "generate"

## Retrieve the relevant documents

In [ ]:
def retrieve_documents(state: RagState) -> dict:
    return {"documents": retriever.invoke(state["question"])}

## Prepare readable context with source labels

In [ ]:
def format_context(documents: list[Document]) -> str:
    return "\n\n".join(
        f"[{doc.metadata['source']}]\n{doc.page_content}"
        for doc in documents
    )

## Generate from context or answer directly

In [ ]:
def generate_answer(state: RagState) -> dict:
    if state["needs_retrieval"]:
        instruction = (
            "Use only the context. If it is insufficient, say you could not find the answer. "
            "Cite source labels in brackets.\nContext:\n" + format_context(state["documents"])
        )
    else:
        instruction = "Reply briefly to the user."
    reply = model.invoke([
        SystemMessage(instruction), HumanMessage(state["question"]),
    ])
    return {"answer": reply.text}

## Register the three working nodes

In [ ]:
builder = StateGraph(RagState)
builder.add_node("decide", decide_retrieval)
builder.add_node("retrieve", retrieve_documents)
builder.add_node("generate", generate_answer)

## Connect the retrieval branch

In [ ]:
builder.add_edge(START, "decide")
builder.add_conditional_edges(
    "decide", should_retrieve,
    {"retrieve": "retrieve", "generate": "generate"},
)
builder.add_edge("retrieve", "generate")
builder.add_edge("generate", END)
graph = builder.compile()

## Start each question with fresh state

In [ ]:
def ask_question(question: str) -> RagState:
    return graph.invoke({
        "question": question, "documents": [],
        "answer": "", "needs_retrieval": False,
    })

## Inspect a retrieval question

In [ ]:
result = ask_question("How does RAG use retrieved documents?")
print("Retrieved:", result["needs_retrieval"])
print("Sources:", [doc.metadata["source"] for doc in result["documents"]])
print("Answer:", result["answer"])

## Try the direct path

In [ ]:
greeting = ask_question("Hello!")
print("Retrieved:", greeting["needs_retrieval"])
print("Documents:", greeting["documents"])
print("Answer:", greeting["answer"])